In [1]:
import socket
import json
import time
import math
import numpy as np
import yaml
# 自定义库
from arm5dof_uservo import Arm5DoFUServo
from arm_test import ArmApplication
from config import *
# 创建机械臂
arm = Arm5DoFUServo(is_debug=True)
# 创建机械臂应用
arm_app = ArmApplication(arm, is_debug=True)


In [2]:
import socket
import json
import time
import math
import threading

# ---- 参数 ----
TURN_STEP_RAD  = math.radians(3.0)
PAN_RAD_MIN    = math.radians(-90.0)
PAN_RAD_MAX    = math.radians(90.0)
TURN_T         = 0.3
MOVE_INTERVAL  = 0.1
last_move_time = 0

# ---- pan_rad 初始化 ----
if "pan_rad" not in globals():
    pan_rad = math.radians(0)

# ---- 共享状态 ----
latest_cmd   = None
cmd_lock     = threading.Lock()
catch_busy   = False      # 只有 catch 执行时为 True


# ---- 指令处理 ----
def handle_command(cmd):
    global pan_rad, last_move_time, arm, arm_app

    action = cmd.get("action")
    now = time.time()

    # stop
    if action == "stop":
        print("[CMD] stop → 回归 home")
        arm_app.home()
        try:
            angles = arm.get_servo_angle_list()
            pan_rad = math.radians(angles[0])
        except Exception:
            pan_rad = math.radians(88.0)
        last_move_time = now
        return

    # hold
    if action == "hold":
        return

    # target1 / target2
    if action in ("target1", "target2"):
        if now - last_move_time < MOVE_INTERVAL:
            return
        last_move_time = now

        if action == "target1":
            pan_rad += TURN_STEP_RAD
        else:
            pan_rad -= TURN_STEP_RAD

        pan_rad = max(PAN_RAD_MIN, min(PAN_RAD_MAX, pan_rad))
        arm.set_joint_angle_soft("joint1", pan_rad, T=TURN_T)
        print(f"[CMD] {action} → pan={math.degrees(pan_rad):.1f}°")
        return

    # catch
    if action == "catch":
        print("[CMD] catch → 抓取当前位置")
        try:
            x, y, z, pitch, roll = arm_app.arm.get_tool_pose()
            wx, wy, _ = arm_app.tf_arm2ws(x, y, z)
            wz = arm_app.object_height-5
            print(f"[DEBUG] 抓取点 ws=({wx:.1f},{wy:.1f},{wz:.1f})")
            arm_app.grab_move(wx, wy, wz)
            print("[CMD] catch → 完成")
        except Exception as e:
            print(f"[CMD] catch 失败: {e}")
        return

    print(f"[Jupyter] 未知动作: {action}")


# ---- 执行线程 ----
def executor_loop():
    global latest_cmd, catch_busy

    while True:
        with cmd_lock:
            cmd = latest_cmd
            latest_cmd = None

        if cmd is not None:
            action = cmd.get("action")

            if action == "catch":
                if catch_busy:
                    time.sleep(0.01)
                    continue
                catch_busy = True
                try:
                    handle_command(cmd)
                except Exception as e:
                    print("[Jupyter] 执行失败:", e)
                finally:
                    catch_busy = False
            else:
                try:
                    handle_command(cmd)
                except Exception as e:
                    print("[Jupyter] 执行失败:", e)

        time.sleep(0.01)


# ---- 服务端 ----
def run_server():
    global latest_cmd

    HOST = "0.0.0.0"
    PORT = 9999

    srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    srv.bind((HOST, PORT))
    srv.listen(1)
    print(f"[Jupyter] 监听 {HOST}:{PORT} ...")

    t = threading.Thread(target=executor_loop, daemon=True)
    t.start()

    while True:
        conn, addr = srv.accept()
        print(f"[Jupyter] 客户端接入: {addr}")
        buf = b""
        try:
            while True:
                data = conn.recv(4096)
                if not data:
                    break
                buf += data

                lines = buf.split(b"\n")
                buf = lines[-1]
                complete = [l for l in lines[:-1] if l.strip()]

                if complete:
                    line = complete[-1]
                    try:
                        cmd = json.loads(line.decode("utf-8"))
                        with cmd_lock:
                            latest_cmd = cmd      # 永远覆盖
                    except Exception as e:
                        print("[Jupyter] 解析失败:", e, line)
        except Exception as e:
            print("[Jupyter] 连接异常:", e)
        finally:
            conn.close()
            print("[Jupyter] 客户端断开")

In [ ]:
run_server()

[Jupyter] 监听 0.0.0.0:9999 ...
[Jupyter] 客户端接入: ('192.168.55.100', 59806)
[CMD] catch → 抓取当前位置
[DEBUG] 抓取点 ws=(-22.0,-2.5,23.0)
运动到 wx=-21.98, wy=-2.54, wz=78.00
运动到 arm_x=154.97, arm_y=-3.67, wz=62.69
运动到 wx=-21.98, wy=-2.54, wz=28.00
运动到 arm_x=154.97, arm_y=-3.67, wz=12.69
[CMD] catch → 完成
[CMD] target2 → pan=-3.0°
[CMD] target2 → pan=-6.0°
[CMD] target2 → pan=-9.0°
[CMD] target2 → pan=-12.0°
[CMD] target2 → pan=-15.0°
[CMD] stop → 回归 home
[CMD] target1 → pan=3.8°
[CMD] target1 → pan=6.8°
[CMD] target1 → pan=9.8°
[CMD] target1 → pan=12.8°
[CMD] target1 → pan=15.8°
[CMD] target1 → pan=18.8°
[CMD] target1 → pan=21.8°
[CMD] target1 → pan=24.8°
[CMD] target1 → pan=27.8°
[CMD] target1 → pan=30.8°
[CMD] target2 → pan=27.8°
[CMD] target2 → pan=24.8°
[CMD] target2 → pan=21.8°
[CMD] target2 → pan=18.8°
[CMD] target2 → pan=15.8°
[CMD] target2 → pan=12.8°
[CMD] target2 → pan=9.8°
[CMD] target2 → pan=6.8°
[CMD] target2 → pan=3.8°
[CMD] target2 → pan=0.8°
[CMD] target2 → pan=-2.2°
[CMD] target2 → 